# 3교시 실습 — 지저분한 데이터 길들이기 🧹

> **사용법**: 🔵 **수업**(함께 실행) · 🟡 **셀프**(직접 채우기 `____`).
> 정답본(`_solution`)에는 **실행 결과 + 💡해석**이 셀마다 달려 있습니다.

---

## 🗂 오늘의 미션

분석가에게 오는 데이터는 교과서처럼 깨끗하지 않습니다. **현실은 지저분합니다.**
오늘 두 개의 진짜 데이터를 정제합니다.

1. **🏠 부동산** — 국토교통부 아파트 실거래가 (글자로 된 금액, 빈 칸 가득)
2. **💰 금융** — 금(Gold) 시세 시계열 (수집이 끊긴 구멍들)

특히 **결측치(빈 칸)** 를 *눈으로 보고*, **여러 방법으로 메우는 법**을 배웁니다 —
버리기부터 시작해서, 마지막엔 **AI가 쓰는 알고리즘 방식**까지.

> 💡 분석가는 시간의 절반 이상을 '정제'에 씁니다. 화려하진 않지만, 여기서 분석의 품질이 갈립니다.

## 0. 준비  🔵

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, urllib.request
from matplotlib import font_manager as fm

DATA_BASE = "https://raw.githubusercontent.com/acho98/gunyang-data/main/"
def set_korean_font():
    cands = ["../data/BMDOHYEON.ttf", "data/BMDOHYEON.ttf", "BMDOHYEON.ttf"]
    if not any(os.path.exists(p) for p in cands):
        try: urllib.request.urlretrieve(DATA_BASE + "BMDOHYEON.ttf", "BMDOHYEON.ttf")
        except Exception: pass
    for p in cands:
        if os.path.exists(p):
            try:
                fm.fontManager.addfont(p)
                plt.rcParams["font.family"] = fm.FontProperties(fname=p).get_name()
                plt.rcParams["axes.unicode_minus"] = False; return
            except Exception: pass
    print("(안내) 한글 폰트를 못 찾았어요. 영문은 정상입니다.")
set_korean_font()
print("준비 완료!")

# Part A. 🏠 부동산 실거래가 정제

> 부장님: *"국토부에서 받은 2021년 5월 전국 아파트 실거래가야. 평균 가격이랑 비싼 동네 좀 뽑아줘. 근데... 데이터가 좀 지저분할 거야."*

### A-1. 입수하고 첫인상 보기  🔵

In [ ]:
apt = pd.read_csv(DATA_BASE + "apt_realprice_202105.csv")
print("거래 건수:", len(apt))
apt.head()

🤔 **확인**: 몇 건의 거래가 있나요? 어떤 정보들이 기록돼 있는지 살펴보세요.

### A-2. 타입 점검 — `info()`  🟡
> 💡 힌트: `apt.info()`

In [ ]:
# TODO: 각 컬럼의 타입과 결측 확인
apt.____()

🤔 **직접 해석**: `거래금액(만원)`의 타입(Dtype)은 무엇인가요? Non-Null 개수가 유난히 적은 컬럼은 무엇인가요?

### A-3. 문제① 글자로 된 금액 → 숫자  🔵
`거래금액`이 `"19,500"` 처럼 **쉼표가 든 글자**라 계산이 안 됩니다. 쉼표를 지우고 숫자로 바꿉니다. (핵심 기법!)

In [ ]:
print("바꾸기 전:", apt["거래금액(만원)"].iloc[0], "/ 타입:", apt["거래금액(만원)"].dtype)
apt["거래금액(만원)"] = apt["거래금액(만원)"].str.replace(",", "").astype(int)
print("바꾼 후 :", apt["거래금액(만원)"].iloc[0], "/ 타입:", apt["거래금액(만원)"].dtype)
print("이제 평균이 구해진다 → {:,.0f}만원".format(apt["거래금액(만원)"].mean()))

🤔 **확인**: 바꾸기 전과 후의 '타입'이 어떻게 달라졌나요? 왜 글자 상태에선 평균을 못 구할까요?

## 🔍 결측치(빈 칸) 다루기 — 오늘의 핵심

데이터에 비어 있는 칸(`NaN`)을 **결측치**라고 합니다. 분석가는 이걸 ① **눈으로 확인**하고 ② **방법을 골라 처리**합니다.

### A-4. 결측 탐색 — 어디에 얼마나?  🟡
> 💡 힌트: `apt.isnull().sum()`

In [ ]:
# TODO: 컬럼별 결측치 개수 (많은 순 정렬)
apt.isnull().____().sort_values(ascending=False)

🤔 **직접 해석**: 결측치가 가장 많은 컬럼은? 거의 다 비어있는 컬럼이 보이나요?

### A-5. 결측 시각화 ① — 막대그래프  🔵
결측 개수를 막대로 그리면 '얼마나 심각한지'가 한눈에 보입니다.

In [ ]:
miss = apt.isnull().sum().sort_values(ascending=False)
miss = miss[miss > 0]                      # 결측이 있는 컬럼만
miss.plot(kind="bar", color="#C0392B")
plt.title("컬럼별 결측치 개수"); plt.ylabel("빈 칸 수"); plt.xticks(rotation=30, ha="right")
plt.tight_layout(); plt.show()

🤔 **확인**: 어떤 컬럼의 막대가 압도적으로 높나요? 이 컬럼은 채우는 게 나을까요, 버리는 게 나을까요?

### A-6. 결측 시각화 ② — 히트맵(지도)  🔵
어느 '위치'가 비었는지 지도처럼 봅니다. (밝은 칸 = 빈 칸)

In [ ]:
sample = apt.sample(60, random_state=1)     # 60건만 뽑아서
sns.heatmap(sample.isnull(), cbar=False, cmap="Reds")
plt.title("결측 지도 (밝은 칸 = 빈 칸)"); plt.tight_layout(); plt.show()

🤔 **확인**: 어느 열이 통째로 빨간가요? 결측이 무작위로 흩어져 있나요, 한 컬럼에 몰려 있나요?

## 결측 처리 방법 — 상황에 맞게 고른다

| 방법 | 언제 |
|---|---|
| ① 컬럼 버리기 | 거의 다 비었을 때 |
| ② 행 버리기 | 결측이 아주 적을 때 |
| ③ 대표값 채우기 | 평균·중앙값·최빈값으로 |
| ④ 앞뒤값/보간 | 시계열(시간순) 데이터 |
| ⑤ 알고리즘(KNN) | 여러 변수를 함께 보고 추정 |

Part A에선 ①②③, Part B에서 ④, Part C에서 ⑤를 직접 해봅니다.

> **💉 의료 데이터 한 스푼 — 결측·이상치는 더 조심!**
> 병원 데이터에선 '검사를 안 받은 것'도 결측인데, 이게 **무작위가 아닐 때가 많아요** — 건강해서 안 받았거나, 너무 위중해서 못 받았거나. 즉 **결측 자체가 정보**일 수 있죠. 그래서 무조건 평균으로 채우기 전에 *"왜 비었나"* 를 먼저 생각해야 합니다.
> 이상치도 마찬가지 — 혈압 200을 '이상치'라고 무조건 지우면 **진짜 중증 환자를 버리는 셈**입니다. *오류인지 진짜 신호인지 확인 후* 판단하세요. (자세한 건 5교시 '의료 데이터를 분석할 때'에서!)

### A-7. 방법① 거의 빈 컬럼 버리기  🟡
`해제사유발생일`은 97.7%가 비었으니 통째로 버립니다.
> 💡 힌트: `apt.drop(columns=["해제사유발생일"])`

In [ ]:
# TODO: '해제사유발생일' 컬럼 통째로 버리기
apt = apt.____(columns=["해제사유발생일"])
print("남은 컬럼 수:", apt.shape[1])

🤔 **확인**: 컬럼이 몇 개로 줄었나요?

### A-8. 방법② 결측 적은 행 버리기  🟡
`번지`·`본번` 등은 결측이 몇 개뿐 → 그 행만 버려도 손실이 거의 없습니다.
> 💡 힌트: `apt.dropna()`

In [ ]:
# TODO: 결측이 있는 행 제거
before = len(apt)
apt = apt.____()
print(f"{before}건 → {len(apt)}건 (버린 행: {before - len(apt)}건)")

🤔 **직접 해석**: 몇 건이 버려졌나요? 전체 대비 많은 편인가요, 무시할 만한가요?

### A-9. 방법③ 대표값으로 채우기 (평균 vs 중앙값)  🔵
이번엔 일부러 결측을 만들어 봅니다 — *"몇몇 거래는 금액이 누락됐다"* 는 상황.
그리고 **평균/중앙값**으로 채워 비교합니다. (1교시 '평균의 함정' 복습!)

In [ ]:
# [왜 일부러 비우나?] 이 데이터엔 '거래금액' 결측이 거의 없어서,
#  대표값(평균/중앙값)으로 채우는 연습을 할 '빈 칸'이 없습니다.
#  그래서 일부러 2000건을 비워 "몇몇 거래의 금액이 누락된" 실무 상황을 재현합니다.
#  (실무 데이터엔 처음부터 이런 결측이 들어 있어요 — 여기선 연습용으로 만드는 것!)
rng = np.random.default_rng(0)
apt2 = apt.copy()
idx = rng.choice(apt2.index, size=2000, replace=False)   # 2000건의 금액을 일부러 비움
apt2.loc[idx, "거래금액(만원)"] = np.nan
print("일부러 만든 결측:", apt2["거래금액(만원)"].isnull().sum(), "건")
print("평균:   {:,.0f}만원".format(apt2["거래금액(만원)"].mean()))
print("중앙값: {:,.0f}만원".format(apt2["거래금액(만원)"].median()))

🤔 **직접 해석**: 평균과 중앙값 중 어느 게 더 큰가요? 집값 데이터에서 결측을 채운다면 어느 값이 더 현실적일까요?

### A-10. 중앙값으로 채우기  🟡
> 💡 힌트: `apt2["거래금액(만원)"].fillna( ... )` 에 중앙값을 넣기

In [ ]:
# TODO: 거래금액 결측을 '중앙값'으로 채우기
med = apt2["거래금액(만원)"].median()
apt2["거래금액(만원)"] = apt2["거래금액(만원)"].____(med)
print("남은 결측:", apt2["거래금액(만원)"].isnull().sum(), "→ 0이면 성공")

🤔 **확인**: 남은 결측이 0이 됐나요? 만약 '지역' 같은 범주형이라면 평균 대신 무엇으로 채워야 할까요?

### A-11. 정제 끝 → 분석 맛보기  🟡
깨끗해진 데이터로 **시도별 평균 거래금액 top 5**를 뽑아보세요. (`시군구`의 맨 앞이 시·도)
> 💡 힌트: `apt["시군구"].str.split().str[0]` 로 시·도 추출 후 groupby

In [ ]:
# TODO: 시도별 평균 거래금액 top 5
apt["시도"] = apt["시군구"].str.split().str[0]
(apt.groupby("시도")["거래금액(만원)"].____().sort_values(ascending=False).head(5).round(0))

🤔 **직접 해석**: 평균 거래금액이 가장 높은 시·도는 어디인가요? 예상과 맞나요?

# Part B. 💰 금융 시세 — 시계열의 결측

> 부장님: *"이번엔 금 시세 데이터야. 그래프 그려서 흐름 좀 보자. 근데 수집 서버가 가끔 멈춰서 며칠씩 비었을 거야."*

**시계열**(시간 순서 데이터)의 결측은 부동산과 다르게 처리합니다 — *앞뒤 값*이나 *보간*을 써요.

### B-1. 입수 + 시계열 만들기  🔵

In [ ]:
gold = pd.read_csv(DATA_BASE + "gold_5y.csv")    # 컬럼: ds(날짜), y(가격)
gold["ds"] = pd.to_datetime(gold["ds"])
gold = gold.sort_values("ds").reset_index(drop=True)
print("기간:", gold["ds"].min().date(), "~", gold["ds"].max().date(), "/", len(gold), "행")
gold.head()

🤔 **확인**: 데이터의 기간(시작~끝)과 행 수는? 컬럼은 무엇인가요?

### B-2. 결측 만들기 — 수집이 끊긴 구멍  🔵
*"서버가 멈춰 며칠 데이터가 빈"* 상황을 재현합니다.

In [ ]:
# [왜 일부러 비우나?] 시계열 채우기(앞뒤 값·보간)를 연습하려면 '구멍'이 필요합니다.
#  실무에서 서버 장애로 며칠치 데이터가 빠지는 상황을 재현해, 일부러 구멍을 냅니다.
g = gold.tail(120).copy().reset_index(drop=True)   # 최근 120일만
g.loc[40:48, "y"] = np.nan      # 9일치 구멍
g.loc[80:83, "y"] = np.nan      # 4일치 구멍
print("구멍(결측):", g["y"].isnull().sum(), "일치")

🤔 **확인**: 며칠치를 비웠나요? 시계열에 구멍이 생기면 어떤 문제가 있을까요?

### B-3. 결측 시각화 — 구멍 난 그래프  🟡
결측이 있는 채로 선그래프를 그리면 **구멍(끊김)** 이 보입니다.
> 💡 힌트: `plt.plot(g["ds"], g["y"], marker="o")`

In [ ]:
# TODO: 구멍 난 시세를 선그래프로 (marker로 점도 표시)
plt.figure(figsize=(10, 4))
plt.plot(g["ds"], g["____"], marker="o", markersize=3)
plt.title("금 시세 — 구멍 난 구간(끊긴 곳 = 결측)"); plt.ylabel("가격")
plt.tight_layout(); plt.show()

🤔 **확인**: 그래프에서 선이 끊긴 곳이 보이나요? 그곳이 바로 결측 구간입니다.

### B-4. 방법④-a 앞/뒤 값으로 채우기 (ffill/bfill)  🔵
시계열은 *"직전 값"* 으로 채우는 게 자연스러울 때가 많습니다 (어제 가격 ≈ 오늘 가격).

In [ ]:
g_ffill = g.copy()
g_ffill["y"] = g_ffill["y"].ffill()     # forward fill: 바로 앞 값으로 채움
print("ffill 후 결측:", g_ffill["y"].isnull().sum())

🤔 **확인**: ffill 후 결측이 0이 됐나요? '앞 값으로 채운다'가 가격 데이터에 자연스러운 이유는?

### B-5. 방법④-b 보간(interpolate) — 부드럽게 잇기  🟡
구멍의 **양 끝을 직선으로 이어** 자연스럽게 채웁니다. 시계열에 가장 매끄러운 방법.
> 💡 힌트: `g["y"].interpolate()`

In [ ]:
# TODO: 보간(interpolate)으로 결측 채우기
g_interp = g.copy()
g_interp["y"] = g_interp["y"].____()
print("보간 후 결측:", g_interp["y"].isnull().sum())

🤔 **확인**: 보간 후 결측이 0인가요? ffill(계단식)과 보간(직선식)은 채우는 모양이 어떻게 다를까요?

### B-6. before / after 비교 시각화  🔵
원본(구멍)·ffill·보간을 한 그림에 겹쳐 봅니다.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(g_ffill["ds"], g_ffill["y"], "-", color="orange", label="④-a ffill (계단식)")
plt.plot(g_interp["ds"], g_interp["y"], "-", color="green", label="④-b 보간 (부드럽게)")
plt.plot(g["ds"], g["y"], "o", color="black", markersize=3, label="원본(있는 값)")
plt.title("결측 채우기: ffill vs 보간"); plt.ylabel("가격"); plt.legend()
plt.tight_layout(); plt.show()

🤔 **직접 해석**: 구멍 구간에서 주황선과 초록선의 모양이 어떻게 다른가요? 가격 흐름엔 어느 쪽이 더 그럴듯한가요?

# Part C. 🤖 마지막 — 알고리즘으로 채우기 (KNN)

지금까지는 *한 컬럼만 보고* 채웠습니다(평균, 앞값 등).
하지만 **여러 정보를 함께 보면 더 똑똑하게** 채울 수 있어요.

> 예: 어떤 아파트의 '가격'이 비었다면 → **면적·층·건축년도가 비슷한 다른 아파트들**의 가격을 보고 추정하면 되겠죠?

이게 **KNN(K-최근접 이웃) 대치** 입니다. *"비슷한 이웃들의 값으로 채운다."* — AI/머신러닝에서 쓰는 방법이에요.

### C-1. 작은 샘플 + 결측 만들기  🔵
KNN은 계산이 무거워 **작은 샘플(200건)** 의 수치 컬럼으로 시연합니다.

In [ ]:
# [왜 일부러 비우나?] KNN으로 '채우는' 연습을 하려면 채울 빈 칸이 있어야 합니다.
#  그래서 200건 중 15건의 가격을 일부러 비워, "이웃(비슷한 집)을 보고 값을 추정"하는 상황을 만듭니다.
cols = ["전용면적(㎡)", "거래금액(만원)", "층", "건축년도"]
s = apt[cols].sample(200, random_state=42).reset_index(drop=True)
rng = np.random.default_rng(7)
miss_idx = rng.choice(s.index, size=15, replace=False)
s.loc[miss_idx, "거래금액(만원)"] = np.nan      # 15건의 가격을 비움
print("결측 만든 가격:", s["거래금액(만원)"].isnull().sum(), "건")
s.head()

🤔 **확인**: 몇 건의 가격을 비웠나요? KNN이 가격을 채울 때 참고할 '다른 정보'는 무엇인가요?

### C-2. KNN 대치 적용  🔵
`scikit-learn`(코랩 기본 탑재)의 `KNNImputer`. *비슷한 이웃 5명의 평균*으로 빈 칸을 채웁니다.

In [ ]:
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

# KNN은 '거리'로 이웃을 찾으므로, 단위가 다른 컬럼은 스케일을 맞춰줍니다(스케일링)
scaler = StandardScaler()
scaled = scaler.fit_transform(s)                          # 결측(NaN)은 그대로 유지
imputer = KNNImputer(n_neighbors=5)
with np.errstate(all="ignore"):                                   # sklearn 내부 계산의 무해한 경고 숨김
    filled = scaler.inverse_transform(imputer.fit_transform(scaled))  # 채운 뒤 원래 단위로 복원
s_knn = pd.DataFrame(filled, columns=cols)
print("KNN 후 결측:", int(s_knn["거래금액(만원)"].isnull().sum()))
print("평균으로만 채웠다면 모두 같은 값이지만, KNN은 행마다 다르게 채웁니다 ↓")
s_knn.loc[miss_idx, "거래금액(만원)"].round(0).head()

🤔 **확인**: KNN으로 채운 값들이 서로 같나요, 다른가요? 평균으로 채우는 것과 무엇이 다른가요?

### C-3. 평균 채우기 vs KNN 비교 시각화  🔵

In [ ]:
mean_val = s["거래금액(만원)"].mean()
knn_vals = s_knn.loc[miss_idx, "거래금액(만원)"]

plt.figure(figsize=(9, 4))
plt.scatter(range(len(knn_vals)), knn_vals, color="green", label="KNN (이웃 기반, 제각각)", zorder=3)
plt.axhline(mean_val, color="red", ls="--", label=f"평균 채우기 (모두 {mean_val:,.0f})")
plt.title("결측 채우기: 평균(빨강 직선) vs KNN(초록 점)")
plt.ylabel("채워진 거래금액(만원)"); plt.xlabel("결측이었던 거래 (15건)"); plt.legend()
plt.tight_layout(); plt.show()

🤔 **직접 해석**: 빨간 선(평균)과 초록 점(KNN) 중 어느 쪽이 '집마다 다른 사정'을 반영하나요? 언제 KNN이 평균보다 나을까요?

## 🎯 정리 — 결측치 처리, 이렇게 고른다

| 방법 | 어떻게 | 언제 쓰나 | 오늘 사용 |
|---|---|---|---|
| ① 컬럼 버리기 | `drop(columns=...)` | 거의 다 비었을 때 | 해제사유발생일 |
| ② 행 버리기 | `dropna()` | 결측이 아주 적을 때 | 번지·본번 |
| ③ 대표값 채우기 | `fillna(중앙값/평균/최빈값)` | 일반적, 빠르게 | 거래금액(중앙값) |
| ④ 앞뒤값 / 보간 | `ffill()` / `interpolate()` | **시계열**(시간순) | 금 시세 |
| ⑤ 알고리즘(KNN) | `KNNImputer` | 변수 간 관계 활용 | 가격 추정 |

**핵심**: 결측 처리에 '정답'은 없습니다. **데이터의 성격(범주형/수치형/시계열)** 과 **결측의 양**을 보고 **사람이 판단**해 고릅니다. — 1교시의 그 메시지, 기억나죠?

> 🏠 **셀프스터디**: 이 노트북의 부동산 데이터로 ③을 '평균'으로 바꿔보거나, 금융 데이터의 보간을 `method="spline"` 으로 바꿔보세요. 결과가 어떻게 달라지는지 직접 비교!